# 🎵 NINE-Music — Treinador GPU (Colab, autônomo)

Este notebook **já contém o código** (`nine_upgrade`), então não precisa de GitHub nem de upload.

**Antes:** *Ambiente de execução → Alterar tipo de ambiente → GPU (T4).* Depois rode as células em ordem.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execucao > Alterar tipo.'

## 1. Criar o código (embutido neste notebook)

In [ ]:
import base64, os, zipfile, io, sys
PKG = (
    "..."
    "AAoAjAMAAEE1AAAAAA=="
)
os.makedirs("/content/nine-music-upgrade", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PKG))).extractall("/content/nine-music-upgrade")
os.chdir("/content/nine-music-upgrade"); sys.path.insert(0, os.getcwd())
print("✅ nine_upgrade criado em", os.getcwd()); print(os.listdir("nine_upgrade"))

## 2. Dependências

In [ ]:
!pip install -q datasets
!apt-get install -y -q abcmidi fluidsynth fluid-soundfont-gm > /dev/null 2>&1
!python tests/test_upgrade.py

## 3. Dados
`irishman` = dataset grande (HF). **Confira a licença** antes de distribuir o modelo. Só Nottingham: `--sources nottingham`.

In [ ]:
!python -m nine_upgrade.prepare_data --out data --sources nottingham irishman --max_irishman 60000

## 4. Treino
Presets: `small` (~10M) · `base` (~25M, recomendado na T4) · `large` (~85M).
Se faltar memória: `BATCH = 16` e acrescente `--grad_accum 2`. Se o Colab cair: acrescente `--resume`.

In [ ]:
PRESET = "base"      # small | base | large
ITERS  = 8000
BATCH  = 32
OUT    = f"runs/{PRESET}"
!python -m nine_upgrade.train --data data --out {OUT} --preset {PRESET} --iters {ITERS} --batch {BATCH}

## 5. Gerar tunes (só com compassos válidos) + áudio

In [ ]:
STYLE, KEY, NUM = "jig", "D", 3     # jig | reel | waltz | polka | march | slipjig | slide
!python -m nine_upgrade.generate --ckpt {OUT}/best.pt --tok {OUT}/tokenizer.json --style {STYLE} --key {KEY} --num {NUM} --out samples

In [ ]:
import glob, IPython.display as ipd
for w in sorted(glob.glob('samples/*.wav')):
    print(w); print(open(w.replace('.wav','.abc')).read()); ipd.display(ipd.Audio(w))

## 6. Salvar o modelo
O Colab apaga tudo quando a sessão expira: salve no Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nine-music && cp -r {OUT} /content/drive/MyDrive/nine-music/
print("salvo em MyDrive/nine-music/")

In [ ]:
from google.colab import files
files.download(f'{OUT}/best.pt'); files.download(f'{OUT}/tokenizer.json')